In [ ]:
print("hi")

In [ ]:
# ==========================================
# EXTERNAL BASELINE EVALUATION (DetectGPT - FIXED)
# ==========================================
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
import random
import warnings

warnings.filterwarnings('ignore')

print("="*60)
print("STARTING EXTERNAL BASELINE EVALUATION (DetectGPT - SAFE MODE)")
print("="*60)

SEED = 999
NUM_VARIATIONS = 100
WINDOW_SIZE = 150 

df = pd.read_csv('data/processed/processed_articles.csv') 

_, df_test = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df['is_AI'])
df_human = df_test[df_test['is_AI'] == 0]
df_ai = df_test[df_test['is_AI'] == 1]
n_samp = min(len(df_human), len(df_ai))

df_sampled = pd.concat([
    df_human.sample(n_samp, random_state=SEED),
    df_ai.sample(n_samp, random_state=SEED)
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Loaded {len(df_sampled)} balanced samples.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Models on {device}...")

tokenizer_gpt = GPT2Tokenizer.from_pretrained("gpt2-medium")
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token 
model_gpt = GPT2LMHeadModel.from_pretrained("gpt2-medium").to(device)
model_gpt.eval()

mask_filler = pipeline(
    "fill-mask", 
    model="distilroberta-base", 
    device=0 if device == "cuda" else -1,
    top_k=1
)

def get_log_likelihood(text):
    inputs = tokenizer_gpt(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    if inputs.input_ids.size(1) < 2: return 0.0
    with torch.no_grad():
        outputs = model_gpt(**inputs, labels=inputs.input_ids)
        return -outputs.loss.item()

def perturb_text_safe(text, mask_prob=0.15):
    words = text.split()
    if len(words) < 10: return text
    
    num_masks = max(1, int(len(words) * mask_prob))
    mask_indices = random.sample(range(len(words)), min(num_masks, 15))
    
    for idx in mask_indices:
        words[idx] = "<mask>"
        
    masked_text = " ".join(words)
    
    try:
        results = mask_filler(masked_text, truncation=True, max_length=500)
        res_list = results if isinstance(results, list) else [results]
        for r in res_list:
            masked_text = masked_text.replace("<mask>", r['token_str'], 1)
        return masked_text.replace("<mask>", "")
    except:
        return text

def process_full_text_sliding_window(text):
    words = text.split()
    chunks = [" ".join(words[i:i+WINDOW_SIZE]) for i in range(0, len(words), WINDOW_SIZE)]
    
    chunk_orig_ll = []
    chunk_pert_ll = []
    
    for chunk in chunks:
        if len(chunk) > 3000:
            chunk = chunk[:3000]
            
        if len(chunk.split()) < 10: continue
        
        orig_ll = get_log_likelihood(chunk)
        chunk_orig_ll.append(orig_ll)
        
        p_lls = []
        for _ in range(NUM_VARIATIONS):
            p_text = perturb_text_safe(chunk)
            p_lls.append(get_log_likelihood(p_text))
            
        chunk_pert_ll.append(np.mean(p_lls) if p_lls else orig_ll)
        
    if not chunk_orig_ll:
        return None
        
    avg_orig_ll = np.mean(chunk_orig_ll)
    avg_pert_ll = np.mean(chunk_pert_ll)
    discrepancy = avg_orig_ll - avg_pert_ll
    
    return avg_orig_ll, avg_pert_ll, discrepancy

print("\nProcessing Samples (Safe Perturbations)...")
features_list = []
valid_y_true = []

for idx, text in enumerate(tqdm(df_sampled['Text'].astype(str).tolist(), desc="Evaluating DetectGPT")):
    try:
        result = process_full_text_sliding_window(text)
        if result is not None:
            features_list.append(result)
            valid_y_true.append(df_sampled['is_AI'].iloc[idx])
    except Exception as e:
        continue

y_true = np.array(valid_y_true)
features_array = np.array(features_list)

print("\n" + "="*60)
print("COMPUTING METRICS (ZERO-SHOT vs SUPERVISED)")
print("="*60)

detectgpt_scores = features_array[:, 2] 
threshold = np.median(detectgpt_scores)
y_pred_zero = (detectgpt_scores >= threshold).astype(int)
acc_zero = accuracy_score(y_true, y_pred_zero)
prec_zero, rec_zero, f1_zero, _ = precision_recall_fscore_support(y_true, y_pred_zero, average='macro', zero_division=0)

clf = LogisticRegression(random_state=SEED)
y_pred_sup = cross_val_predict(clf, features_array, y_true, cv=5)
acc_sup = accuracy_score(y_true, y_pred_sup)
prec_sup, rec_sup, f1_sup, _ = precision_recall_fscore_support(y_true, y_pred_sup, average='macro', zero_division=0)

results = [
    {'Model': 'DetectGPT (Zero-Shot Curvature)', 'Accuracy': acc_zero, 'Precision': prec_zero, 'Recall': rec_zero, 'F1-Score': f1_zero},
    {'Model': 'DetectGPT (Logistic Regression Base)', 'Accuracy': acc_sup, 'Precision': prec_sup, 'Recall': rec_sup, 'F1-Score': f1_sup}
]

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv('results/tables/T11_DetectGPT_Baseline_Results.csv', index=False)
print("="*60)
print("✓ Results saved successfully to results/tables/T11_DetectGPT_Baseline_Results.csv")

STARTING EXTERNAL BASELINE EVALUATION (DetectGPT - SAFE MODE)
Loaded 2918 balanced samples.
Loading Models on cuda...


Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0



Processing Samples (Safe Perturbations)...


Evaluating DetectGPT: 100%|██████████| 2918/2918 [12:00:43<00:00, 14.82s/it]    



COMPUTING METRICS (ZERO-SHOT vs SUPERVISED)
                               Model  Accuracy  Precision   Recall  F1-Score
     DetectGPT (Zero-Shot Curvature)  0.500171   0.250086 0.500000  0.333410
DetectGPT (Logistic Regression Base)  0.858073   0.858172 0.858071  0.858063
✓ Results saved successfully to results/tables/T12_DetectGPT_Baseline_Results.csv
